# REDCap clean-positive 9:1 training + new benchmark evaluation

This notebook trains one single REDCap-updated bi-encoder model and evaluates it on the fixed/new benchmark setup.

Main design:

1. Start from the Fan previous bi-encoder checkpoint.
2. Load clean NCIt training pairs.
3. Extract and filter REDCap synthetic clean-positive pairs.
4. Increase REDCap signal with `CLEAN_TO_REDCAP_RATIO = 9`.
5. Train for half an epoch with LR `5e-7`, batch size `64`, and anti-drift weight `0.10` against the initialization checkpoint.
6. Evaluate the newly trained model on the same fixed benchmark query rows and target pool used by the previous rerun.
7. Compare the new model against saved historical results for REDCap v3, BGE raw, OpenAI raw, and CTDS base raw.

The historical BGE/OpenAI/CTDS baseline rows are reused from the previous fixed-benchmark rerun. This is appropriate as long as the benchmark rows, query representation, target representation, target pool, and evaluation logic are unchanged.

In [ ]:
# Cell 1 — Configuration
from pathlib import Path
import datetime as _dt

# -----------------------------
# Training input paths
# -----------------------------
BASE_MODEL_NAME = "uc-ctds/bge-large-en-v1.5-bio-mapping"

# Start from Fan previous checkpoint, same lineage as the earlier REDCap v3 notebook.
INIT_CHECKPOINT_PATH = Path(
    "/opt/gpudata/fan1/heal_cde/ncit/experiments/"
    "2stage_cleaned_multipos/biencoder/biencoder_lora_final.pt"
)

CLEAN_FULL_SPLIT_PATH = Path(
    "/opt/gpudata/fan1/heal_cde/ncit/experiments/"
    "nci_redcap_augmented_cleaned_multipos/"
    "cleaned_multipos_full_split_reconstructed_exact_notebook.csv"
)

AUGMENTED_TRAIN_PLUS_REDCAP_PATH = Path(
    "/opt/gpudata/fan1/heal_cde/ncit/experiments/"
    "nci_redcap_augmented_cleaned_multipos/"
    "cleaned_multipos_train_only_plus_nci_redcap_5000_fixed.csv"
)

# -----------------------------
# Previous fixed/new benchmark rerun output dir
# -----------------------------
# Historical BGE/OpenAI/CTDS/REDCap-v3 results are read from this previous rerun.
# This is valid only if benchmark rows, query representation, target representation,
# target pool, and evaluation logic are unchanged.
PREVIOUS_RERUN_DIR = None  # set to Path("/opt/gpudata/...") if auto-discovery fails
PREVIOUS_RERUN_GLOB = "/opt/gpudata/fan1/heal_cde/ncit/experiments/fan_redcap_hdp00429_fixed_full_rerun_*"

# Required previous files:
#   final_overall_model_comparison.csv OR full_rerun_metrics_per_row.csv
#   benchmark_query_texts_embedded.csv
#   target_texts_embedded.csv

# -----------------------------
# Training controls for this run
# -----------------------------
MIX_RATIO_NAME = "9to1"
CLEAN_TO_REDCAP_RATIO = 9
TARGETED_REDCAP_MIN_FRACTION = 0.60

LEARNING_RATE = 5e-7
TEMPERATURE = 0.05
MAX_EPOCH_FRACTION = 0.5
PER_DEVICE_BATCH_SIZE = 64
ENCODE_BATCH_SIZE = 128
GRAD_ACCUM_STEPS = 1
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY = 0.01
SEED = 42

# Anti-drift against the initialization checkpoint.
# This keeps the updated model close to the previous Fan checkpoint while adding REDCap signal.
ANTI_DRIFT_DISTILL_WEIGHT = 0.10

RUN_ID = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_ROOT = Path("/opt/gpudata/fan1/heal_cde/ncit/experiments/redcap_cleanpos_9to1_single_model")
RUN_NAME = (
    f"{RUN_ID}_{MIX_RATIO_NAME}_lr{LEARNING_RATE:g}_"
    f"redcapcleanpos_antidrift{ANTI_DRIFT_DISTILL_WEIGHT:g}_epochfrac{MAX_EPOCH_FRACTION:g}_bs{PER_DEVICE_BATCH_SIZE}"
)
OUT_DIR = OUT_ROOT / RUN_NAME
DATA_DIR = OUT_DIR / "data"
CKPT_DIR = OUT_DIR / "biencoder"
EVAL_DIR = OUT_DIR / "eval"
for d in [DATA_DIR, CKPT_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR)
print("INIT_CHECKPOINT_PATH:", INIT_CHECKPOINT_PATH, "exists:", INIT_CHECKPOINT_PATH.exists())
print("CLEAN_FULL_SPLIT_PATH:", CLEAN_FULL_SPLIT_PATH, "exists:", CLEAN_FULL_SPLIT_PATH.exists())
print("AUGMENTED_TRAIN_PLUS_REDCAP_PATH:", AUGMENTED_TRAIN_PLUS_REDCAP_PATH, "exists:", AUGMENTED_TRAIN_PLUS_REDCAP_PATH.exists())
print("CLEAN_TO_REDCAP_RATIO:", CLEAN_TO_REDCAP_RATIO)
print("LEARNING_RATE:", LEARNING_RATE)
print("PER_DEVICE_BATCH_SIZE:", PER_DEVICE_BATCH_SIZE)
print("ENCODE_BATCH_SIZE:", ENCODE_BATCH_SIZE)
print("MAX_EPOCH_FRACTION:", MAX_EPOCH_FRACTION)
print("ANTI_DRIFT_DISTILL_WEIGHT:", ANTI_DRIFT_DISTILL_WEIGHT)

In [ ]:
# Cell 2 — Imports and reproducibility
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import json
import math
import random
import hashlib
from dataclasses import dataclass
from collections import Counter
from typing import Any, Dict, List, Optional, Literal

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device

try:
    from peft import LoraConfig, get_peft_model
except Exception as e:
    raise ImportError("peft is required. Install it in your GPU env if missing: pip install peft") from e

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =====================
# Cell 3 — Text builders and robust column detection
# =====================
# Fixed: the clean/augmented files here use:
#   variable_para            -> query / anchor text
#   alternate_variable_para  -> positive / target text
# The earlier notebook only looked for anchor/positive/query/target columns,
# so it produced 0 training pairs. This version explicitly supports the Fan
# NCIt pair columns.


def clean_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def add_part(parts, label, value):
    value = clean_str(value)
    if value and value.lower() not in {"nan", "none", "null", "[]", "{}"}:
        parts.append(f"{label}: {value}")


def first_nonempty(row: pd.Series, cols):
    for c in cols:
        if c in row.index and clean_str(row[c]):
            return clean_str(row[c])
    return ""


def build_query_text(row: pd.Series) -> str:
    # Most important for this project:
    # clean_full / aug_plus use variable_para as the query text.
    ready = first_nonempty(row, [
        "anchor", "variable_para", "query", "query_text", "input_text", "field_text", "query_norm"
    ])
    if ready:
        return ready

    parts = []
    # Canonical field_code_first_typed-style query.
    query_cols = [
        ("field_name", "field_name"),
        ("field_type", "field_type"),
        ("field_title", "field_title"),
        ("field_enumLabels", "field_enumLabels"),
        ("field_constraints", "field_constraints"),
        ("field_description", "field_description"),
        ("CDE_instrument", "instrument"),
        ("CDE_instrument (file_name)", "instrument"),
        ("instrument", "instrument"),
    ]
    for col, label in query_cols:
        if col in row.index:
            add_part(parts, label, row[col])
    return " | ".join(parts)


def build_target_text(row: pd.Series) -> str:
    # Most important for this project:
    # clean_full / aug_plus use alternate_variable_para as the positive target text.
    ready = first_nonempty(row, [
        "positive", "alternate_variable_para", "target", "target_text", "positive_text", "sde_text", "target_norm"
    ])
    if ready:
        return ready

    parts = []
    # Support both element_* and target_* naming.
    target_cols = [
        ("target_sde_id", "sde_id"),
        ("target_sde_name", "sde_name"),
        ("target_sde_title", "sde_title"),
        ("target_sde_description", "sde_description"),
        ("target_element_name", "element_name"),
        ("target_element_type", "element_type"),
        ("target_element_title", "element_title"),
        ("target_element_description", "element_description"),
        ("element_name", "element_name"),
        ("element_type", "element_type"),
        ("element_title", "element_title"),
        ("element_description", "element_description"),
        ("enumLabels", "enumLabels"),
        ("encoding", "encoding"),
        ("constraints.enum", "constraints.enum"),
        ("standardsMappings.id", "standardsMappings.id"),
        ("standardsMappings.source", "standardsMappings.source"),
        ("CDE_HEAL_ID", "CDE_HEAL_ID"),
        ("nci_code", "nci_code"),
        ("label", "label"),
    ]
    for col, label in target_cols:
        if col in row.index:
            add_part(parts, label, row[col])
    return " | ".join(parts)


def pair_fingerprint(anchor: str, positive: str) -> str:
    s = (anchor.strip().lower() + "\n---\n" + positive.strip().lower()).encode("utf-8")
    return hashlib.md5(s).hexdigest()


def show_columns(df, name):
    print(f"\n{name}: shape={df.shape}")
    print(list(df.columns))

In [ ]:
# =====================
# Cell 4 — Load clean train and augmented train+REDCap data
# =====================
assert CLEAN_FULL_SPLIT_PATH.exists(), f"Missing clean split file: {CLEAN_FULL_SPLIT_PATH}"
assert AUGMENTED_TRAIN_PLUS_REDCAP_PATH.exists(), f"Missing augmented file: {AUGMENTED_TRAIN_PLUS_REDCAP_PATH}"

clean_full = pd.read_csv(CLEAN_FULL_SPLIT_PATH, low_memory=False)
aug_plus = pd.read_csv(AUGMENTED_TRAIN_PLUS_REDCAP_PATH, low_memory=False)

show_columns(clean_full, "clean_full")
show_columns(aug_plus, "aug_plus")

# Keep train split only if split column exists.
split_cols = [c for c in clean_full.columns if c.lower() in {"split", "data_split", "train_val_test"}]
if split_cols:
    split_col = split_cols[0]
    clean_train_raw = clean_full[clean_full[split_col].astype(str).str.lower().eq("train")].copy()
    print(f"Using train split from column {split_col}: {clean_train_raw.shape}")
else:
    clean_train_raw = clean_full.copy()
    print("No split column found. Using all clean_full as clean_train_raw:", clean_train_raw.shape)

# Build anchor/positive texts.
clean_train = clean_train_raw.copy()
clean_train["anchor"] = clean_train.apply(build_query_text, axis=1)
clean_train["positive"] = clean_train.apply(build_target_text, axis=1)
clean_train = clean_train[(clean_train["anchor"].str.len() > 0) & (clean_train["positive"].str.len() > 0)].copy()
clean_train["source_type"] = "clean"
clean_train["pair_fp"] = [pair_fingerprint(a, p) for a, p in zip(clean_train["anchor"], clean_train["positive"])]

aug = aug_plus.copy()
aug["anchor"] = aug.apply(build_query_text, axis=1)
aug["positive"] = aug.apply(build_target_text, axis=1)
aug = aug[(aug["anchor"].str.len() > 0) & (aug["positive"].str.len() > 0)].copy()
aug["pair_fp"] = [pair_fingerprint(a, p) for a, p in zip(aug["anchor"], aug["positive"])]

# Detect REDCap synthetic rows as augmented rows not present in clean train.
clean_fp = set(clean_train["pair_fp"])
redcap_raw = aug[~aug["pair_fp"].isin(clean_fp)].copy()
redcap_raw["source_type"] = "redcap_synthetic_raw"

print("\nclean_train pairs:", clean_train.shape)
print("aug pairs:", aug.shape)
print("detected redcap_raw pairs:", redcap_raw.shape)

clean_train[["anchor", "positive", "source_type", "pair_fp"]].to_csv(DATA_DIR / "clean_train_pairs.csv", index=False)
redcap_raw.to_csv(DATA_DIR / "redcap_synthetic_detected_raw.csv", index=False)
print("Saved clean/redcap raw data to:", DATA_DIR)

display(clean_train[["anchor", "positive"]].head(3))
display(redcap_raw[["anchor", "positive"]].head(3))

In [ ]:
# =====================
# Cell 5 — REDCap clean-positive filter
# =====================
# Filtering goal:
# - keep REDCap rows that are informative and specific
# - remove generic or weak synthetic pairs
# - keep a focused subset of option-like rows so REDCap signal is stronger

GENERIC_TARGET_TERMS = {
    "status", "type", "other", "total", "score", "value", "result", "number", "amount",
    "name", "date", "time", "day", "month", "year", "unknown", "none", "id", "code",
}
CRITICAL_QUALIFIERS = {
    "parent", "child", "children", "pediatric", "adult", "proxy", "self", "caregiver",
    "least", "worst", "average", "current", "past", "last", "previous", "baseline",
    "short", "form", "version", "item", "scale", "subscale", "total",
    "left", "right", "upper", "lower", "bilateral",
}

RACE_OPTION_TERMS = {
    "american indian", "alaska native", "asian", "black", "african american",
    "native hawaiian", "pacific islander", "white", "middle eastern", "north african",
    "unknown", "not reported", "decline", "prefer not", "other race",
}
ETHNICITY_OPTION_TERMS = {
    "hispanic", "latino", "latina", "latinx", "spanish origin",
    "not hispanic", "non hispanic", "unknown", "not reported", "decline", "prefer not",
}
OPTION_CODE_HINTS = {
    "ai_an", "asian", "bl_aa", "black", "white", "nh_pi", "pi", "hi_la", "latino",
    "hispanic", "mena", "unkn", "unknown", "not_rep", "notreported", "decline",
    "prefer", "other",
}
GENERIC_PARENT_TARGET_PHRASES = {
    "race and or ethnicity",
    "race and ethnicity",
    "parent race",
    "race parent",
    "race what is your race",
    "what is your race",
    "ethnicity",
    "hispanic or latino ethnicity",
    "race",
}


def simple_norm(s):
    s = clean_str(s).lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def token_set(s):
    return set(simple_norm(s).split())


def text_has_any(text, phrases):
    st = simple_norm(text)
    return any(simple_norm(p) in st for p in phrases if simple_norm(p))


def get_labeled_value(text, labels):
    """Extract `label: value` from labeled text. Falls back safely for raw variable_para text."""
    text = str(text)
    for label in labels:
        # capture until pipe / semicolon / newline
        m = re.search(rf"{re.escape(label)}\s*:\s*([^|;\n]+)", text, flags=re.I)
        if m:
            return m.group(1).strip()
    return ""


def raw_anchor_text(anchor):
    # If we wrapped text as a labeled field, unwrap it; otherwise use whole anchor.
    v = get_labeled_value(anchor, ["field_text", "query_text", "variable_para"])
    return v if v else clean_str(anchor)


def raw_positive_text(positive):
    v = get_labeled_value(positive, ["element_text", "target_text", "alternate_variable_para"])
    return v if v else clean_str(positive)


def get_field_title_from_anchor(anchor):
    v = get_labeled_value(anchor, ["field_title"])
    if v:
        return v
    # variable_para may already be a compact query string without labels.
    return raw_anchor_text(anchor)[:400]


def get_field_name_from_anchor(anchor):
    v = get_labeled_value(anchor, ["field_name"])
    if v:
        return v
    raw = raw_anchor_text(anchor)
    m = re.search(r"\b[a-zA-Z][a-zA-Z0-9_]*___[a-zA-Z0-9_]+\b", raw)
    if m:
        return m.group(0)
    # common variable-name-looking token near beginning
    m = re.search(r"\b[a-zA-Z][a-zA-Z0-9_]{2,}\b", raw)
    return m.group(0) if m else ""


def get_field_type_from_anchor(anchor):
    v = get_labeled_value(anchor, ["field_type"])
    if v:
        return v
    raw = raw_anchor_text(anchor).lower()
    for t in ["checkbox", "radio", "dropdown", "enum", "integer", "number", "text", "choice"]:
        if t in raw:
            return t
    return ""


def get_field_enum_from_anchor(anchor):
    v = get_labeled_value(anchor, ["field_enumLabels", "field_enumlabels", "enumLabels", "enum"])
    if v:
        return v
    raw = raw_anchor_text(anchor)
    if any(x in raw.lower() for x in ["unchecked", "checked", "choice", "option", "enum"]):
        return raw[:400]
    return ""


def get_field_constraints_from_anchor(anchor):
    return get_labeled_value(anchor, ["field_constraints", "constraints"])


def target_element_name_from_positive(positive):
    v = get_labeled_value(positive, ["element_name", "sde_name", "target_element_name", "target_sde_name", "label"])
    return v if v else ""


def target_title_from_positive(positive):
    for label in ["element_title", "sde_title", "target_element_title", "target_sde_title", "element_name", "sde_name", "label"]:
        v = get_labeled_value(positive, [label])
        if v:
            return v
    return raw_positive_text(positive)[:400]


def has_enum_or_constraints(anchor):
    raw = raw_anchor_text(anchor).lower()
    return (
        bool(get_field_enum_from_anchor(anchor))
        or bool(get_field_constraints_from_anchor(anchor))
        or "___" in raw
        or "checkbox" in raw
        or "radio" in raw
        or "dropdown" in raw
        or "unchecked" in raw
        or "checked" in raw
    )


def has_description(anchor):
    s = str(anchor).lower()
    if "field_description:" in s:
        return len(s.split("field_description:", 1)[1].split("|", 1)[0].strip()) >= 20
    # Raw variable_para often includes enough context even without an explicit description label.
    return len(simple_norm(anchor).split()) >= 12


def extract_option_label_from_query(anchor):
    raw = raw_anchor_text(anchor)
    field_title = get_field_title_from_anchor(anchor)
    field_name = get_field_name_from_anchor(anchor)

    # Most useful pattern: "Race: Asian" or "Ethnicity: Hispanic or Latino".
    for text in [field_title, raw]:
        m = re.search(r"\b(race|ethnicity)\s*[:=\-]\s*([^|;\n]+)", text, flags=re.I)
        if m:
            opt = m.group(2).strip()
            # remove trailing generic boilerplate if any
            opt = re.split(r"\b(field|type|description|enum|constraint)\b", opt, flags=re.I)[0].strip()
            if opt:
                return opt[:120]

    # Generic colon fallback only if the left side looks like a parent concept.
    if ":" in field_title:
        left, right = field_title.split(":", 1)
        if text_has_any(left, {"race", "ethnicity", "gender", "sex", "language", "education"}):
            return right.strip()[:120]

    # Fallback for obvious enum/checkbox naming.
    enum = get_field_enum_from_anchor(anchor)
    if enum:
        return enum[:120]

    # Fallback: suffix after triple underscore.
    if "___" in field_name:
        return field_name.split("___", 1)[1].strip()
    if "___" in raw:
        m = re.search(r"___([a-zA-Z0-9_]+)", raw)
        if m:
            return m.group(1).strip()

    # If the raw query contains a known option phrase, use that.
    raw_norm = simple_norm(raw)
    for phrase in sorted(RACE_OPTION_TERMS | ETHNICITY_OPTION_TERMS, key=len, reverse=True):
        if simple_norm(phrase) in raw_norm:
            return phrase

    return ""


def is_option_level_query(anchor):
    field_title = get_field_title_from_anchor(anchor)
    field_name = get_field_name_from_anchor(anchor)
    field_type = get_field_type_from_anchor(anchor).lower()
    enum = get_field_enum_from_anchor(anchor)
    constraints = get_field_constraints_from_anchor(anchor)
    raw = raw_anchor_text(anchor).lower()
    return (
        "___" in field_name
        or "___" in raw
        or bool(extract_option_label_from_query(anchor))
        or "checkbox" in field_type
        or "radio" in field_type
        or "dropdown" in field_type
        or "checkbox" in raw
        or "radio" in raw
        or "dropdown" in raw
        or bool(enum)
        or bool(constraints)
    )


def classify_targeted_row(row):
    anchor = clean_str(row["anchor"])
    positive = clean_str(row["positive"])

    field_title = get_field_title_from_anchor(anchor)
    field_name = get_field_name_from_anchor(anchor)
    option_label = extract_option_label_from_query(anchor)
    target_name = target_element_name_from_positive(positive)
    target_title = target_title_from_positive(positive)

    combined_query = " ".join([field_name, field_title, option_label, anchor])
    combined_target = " ".join([target_name, target_title, positive])

    is_option = is_option_level_query(anchor)
    is_race = text_has_any(combined_query, {"race"} | RACE_OPTION_TERMS)
    is_ethnicity = text_has_any(combined_query, {"ethnicity"} | ETHNICITY_OPTION_TERMS)
    is_checkbox_enum = is_option and has_enum_or_constraints(anchor)

    # Specific option-like target: Asian/White/Black/Hispanic/etc.
    # Do NOT count the generic word "race" itself as option-like.
    option_terms_for_target = (RACE_OPTION_TERMS | ETHNICITY_OPTION_TERMS | (OPTION_CODE_HINTS - {"race", "other"}))
    option_overlap = token_set(option_label) & token_set(combined_target)
    target_is_option_like = (
        text_has_any(combined_target, option_terms_for_target)
        or bool(option_overlap - {"race", "ethnicity", "parent", "what", "is", "your"})
    )

    target_is_generic_parent = text_has_any(target_title, GENERIC_PARENT_TARGET_PHRASES) and not target_is_option_like

    targeted = is_option and (
        is_race
        or is_ethnicity
        or is_checkbox_enum
    )

    reasons = []
    if targeted:
        if is_race:
            reasons.append("race_option_level")
        if is_ethnicity:
            reasons.append("ethnicity_option_level")
        if is_checkbox_enum:
            reasons.append("checkbox_or_enum_option")
        if target_is_generic_parent:
            reasons.append("generic_parent_positive_risk")
        if target_is_option_like:
            reasons.append("target_option_like")

    return pd.Series({
        "option_label": option_label,
        "is_option_level_query": bool(is_option),
        "is_race_option_query": bool(is_race and is_option),
        "is_ethnicity_option_query": bool(is_ethnicity and is_option),
        "is_checkbox_or_enum_option_query": bool(is_checkbox_enum),
        "target_is_option_like": bool(target_is_option_like),
        "target_is_generic_parent": bool(target_is_generic_parent),
        "is_targeted_option_row": bool(targeted and not target_is_generic_parent),
        "targeted_reason": ";".join(reasons) if reasons else "",
    })


def quality_decision(row):
    anchor = clean_str(row["anchor"])
    positive = clean_str(row["positive"])
    field_title = get_field_title_from_anchor(anchor)
    field_name = get_field_name_from_anchor(anchor)
    target_title = target_title_from_positive(positive)

    target_info = classify_targeted_row(row)
    protect_specific_option_target = (
        bool(target_info["is_option_level_query"])
        and bool(target_info["target_is_option_like"])
        and not bool(target_info["target_is_generic_parent"])
    )

    reasons = []

    ftoks = token_set(field_title)
    ttoks = token_set(target_title)
    qtoks = token_set(anchor)

    # Basic information checks.
    # For raw variable_para, field_title falls back to the full anchor, so this should not remove valid rows.
    if len(simple_norm(field_title)) < 4 and len(simple_norm(field_name)) < 4:
        reasons.append("too_short_field_title_and_name")

    # Generic target filtering.
    # Important: keep specific race/ethnicity option targets such as Unknown/Not reported when the query is option-level.
    if not protect_specific_option_target:
        if len(ttoks) == 1 and list(ttoks)[0] in GENERIC_TARGET_TERMS:
            reasons.append("target_too_generic_single_token")

        if len(ttoks & GENERIC_TARGET_TERMS) >= max(1, len(ttoks) // 2) and len(ttoks) <= 3:
            reasons.append("target_too_generic")

    # If query is extremely short and has no enum/description, it is weak synthetic signal.
    if len(qtoks) < 5 and not has_enum_or_constraints(anchor) and not has_description(anchor):
        reasons.append("query_too_short_no_enum_no_description")

    # Copy-paste check: exact title copied to target and no additional useful context.
    # Do not apply to protected option targets; option text can legitimately match target text, e.g. Asian -> Asian.
    field_norm = simple_norm(field_title)
    target_norm = simple_norm(target_title)
    if (
        not protect_specific_option_target
        and field_norm and target_norm and field_norm == target_norm
        and not has_enum_or_constraints(anchor) and not has_description(anchor)
    ):
        reasons.append("field_target_exact_copy_without_context")

    # Critical qualifier check: if target has qualifier but query does not, row is dangerous.
    # Do not apply this to protected option-level race/ethnicity rows because terms like Unknown/Other are valid options.
    if not protect_specific_option_target:
        target_qual = ttoks & CRITICAL_QUALIFIERS
        query_qual = qtoks & CRITICAL_QUALIFIERS
        missing_qual = target_qual - query_qual
        if missing_qual:
            reasons.append("missing_critical_qualifier_in_query:" + ";".join(sorted(missing_qual)))

    # Very generic REDCap variable names without meaningful title/description.
    if simple_norm(field_name) in GENERIC_TARGET_TERMS and not field_title and not has_description(anchor):
        reasons.append("generic_variable_name_only")

    # New: if query is option-level but positive looks like a generic parent target, drop it.
    if bool(target_info["target_is_generic_parent"]):
        reasons.append("option_query_to_generic_parent_positive_risk")

    keep = len(reasons) == 0
    return keep, " | ".join(reasons) if reasons else "keep"

if len(redcap_raw) == 0:
    raise ValueError("No REDCap synthetic rows detected. Check augmented train file and clean split file.")

target_info = redcap_raw.apply(classify_targeted_row, axis=1)
redcap_tagged = pd.concat([redcap_raw.reset_index(drop=True), target_info.reset_index(drop=True)], axis=1)

qdec = redcap_tagged.apply(quality_decision, axis=1)
redcap_filtered = redcap_tagged.copy()
redcap_filtered["keep_redcap_v3"] = [x[0] for x in qdec]
redcap_filtered["filter_reason"] = [x[1] for x in qdec]

filter_summary = (
    redcap_filtered["filter_reason"]
    .value_counts()
    .rename_axis("filter_reason")
    .reset_index(name="n")
)
filter_summary["percentage"] = filter_summary["n"] / len(redcap_filtered) * 100

target_summary = (
    redcap_filtered
    .groupby(["keep_redcap_v3", "is_targeted_option_row", "targeted_reason"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

redcap_keep = redcap_filtered[redcap_filtered["keep_redcap_v3"]].copy()
redcap_drop = redcap_filtered[~redcap_filtered["keep_redcap_v3"]].copy()
redcap_targeted_keep = redcap_keep[redcap_keep["is_targeted_option_row"]].copy()

print("redcap_raw:", len(redcap_raw))
print("redcap_keep:", len(redcap_keep))
print("redcap_drop:", len(redcap_drop))
print("keep rate:", len(redcap_keep) / max(len(redcap_raw), 1))
print("targeted option rows kept:", len(redcap_targeted_keep))
print("targeted kept rate among kept:", len(redcap_targeted_keep) / max(len(redcap_keep), 1))

print("\nFilter summary:")
display(filter_summary.head(30))

print("\nTargeted option summary:")
display(target_summary.head(30))

print("\nExamples of targeted option rows kept:")
display(redcap_targeted_keep[[
    "anchor", "positive", "option_label", "targeted_reason",
    "is_race_option_query", "is_ethnicity_option_query",
    "is_checkbox_or_enum_option_query", "target_is_option_like"
]].head(20))

print("\nExamples dropped because option query mapped to generic parent positive:")
display(redcap_drop[redcap_drop["filter_reason"].str.contains("generic_parent", na=False)][[
    "anchor", "positive", "option_label", "targeted_reason", "filter_reason"
]].head(20))

redcap_filtered.to_csv(DATA_DIR / "redcap_synthetic_filter_audit_conservative.csv", index=False)
redcap_keep.to_csv(DATA_DIR / "redcap_synthetic_filtered_keep_conservative.csv", index=False)
redcap_drop.to_csv(DATA_DIR / "redcap_synthetic_filtered_drop_conservative.csv", index=False)
redcap_targeted_keep.to_csv(DATA_DIR / "redcap_synthetic_targeted_option_keep_conservative.csv", index=False)
filter_summary.to_csv(DATA_DIR / "redcap_synthetic_filter_summary_conservative.csv", index=False)
target_summary.to_csv(DATA_DIR / "redcap_synthetic_targeted_option_summary_conservative.csv", index=False)

print("Saved REDCap targeted filter outputs to:", DATA_DIR)
print("STATUS: REDCAP 9TO1 REDCAP FILTER DONE")

In [ ]:
# Cell 6 — Build mixed training data with stronger REDCap clean-positive signal
# Design:
#   clean NCIt data + filtered REDCap clean-positive pairs
#   REDCap cap = len(clean_pairs) // CLEAN_TO_REDCAP_RATIO

assert len(redcap_keep) > 0, "No REDCap rows kept after filtering. Relax filters or inspect audit."

clean_pairs = clean_train[["anchor", "positive", "source_type", "pair_fp"]].drop_duplicates("pair_fp").copy()
clean_pairs["is_targeted_option_row"] = False
clean_pairs["targeted_reason"] = ""

redcap_pairs = redcap_keep[[
    "anchor", "positive", "source_type", "pair_fp",
    "is_targeted_option_row", "targeted_reason",
    "is_race_option_query", "is_ethnicity_option_query", "is_checkbox_or_enum_option_query",
    "option_label"
]].drop_duplicates("pair_fp").copy()
redcap_pairs["source_type"] = np.where(
    redcap_pairs["is_targeted_option_row"],
    "redcap_synthetic_targeted_option_clean_positive",
    "redcap_synthetic_filtered_clean_positive"
)

max_redcap = max(1, len(clean_pairs) // CLEAN_TO_REDCAP_RATIO)

targeted_pool = redcap_pairs[redcap_pairs["is_targeted_option_row"]].copy()
other_pool = redcap_pairs[~redcap_pairs["is_targeted_option_row"]].copy()

targeted_target_n = int(math.ceil(max_redcap * TARGETED_REDCAP_MIN_FRACTION))
targeted_n = min(len(targeted_pool), targeted_target_n)
other_n = max_redcap - targeted_n
other_n = min(len(other_pool), other_n)

targeted_sample = targeted_pool.sample(n=targeted_n, random_state=SEED) if targeted_n > 0 else targeted_pool.head(0)
other_sample = other_pool.sample(n=other_n, random_state=SEED) if other_n > 0 else other_pool.head(0)

remaining = max_redcap - len(targeted_sample) - len(other_sample)
if remaining > 0:
    other_left = other_pool.drop(index=other_sample.index, errors="ignore")
    fill_n = min(len(other_left), remaining)
    if fill_n > 0:
        other_fill = other_left.sample(n=fill_n, random_state=SEED + 1)
        other_sample = pd.concat([other_sample, other_fill], ignore_index=False)

redcap_sample = pd.concat([targeted_sample, other_sample], ignore_index=True)
redcap_sample = redcap_sample.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

train_pairs = pd.concat([clean_pairs, redcap_sample], ignore_index=True)
train_pairs = train_pairs.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("clean_pairs:", len(clean_pairs))
print("redcap_pairs kept available:", len(redcap_pairs))
print("targeted_pool available:", len(targeted_pool))
print("other_pool available:", len(other_pool))
print("max_redcap allowed by ratio:", max_redcap)
print("targeted_sample used:", len(targeted_sample))
print("other_sample used:", len(other_sample))
print("redcap_sample used:", len(redcap_sample))
print("train_pairs total:", len(train_pairs))
print("actual clean:redcap ratio:", len(clean_pairs) / max(len(redcap_sample), 1))
print("actual targeted fraction within REDCap sample:", len(targeted_sample) / max(len(redcap_sample), 1))

display(train_pairs["source_type"].value_counts().rename_axis("source_type").reset_index(name="n"))

source_summary = (
    train_pairs
    .groupby(["source_type"])
    .size()
    .reset_index(name="n")
)
display(source_summary)

train_pairs_path = DATA_DIR / f"training_pairs_clean_redcap_cleanpos_{MIX_RATIO_NAME}.csv"
train_pairs.to_csv(train_pairs_path, index=False)
redcap_sample.to_csv(DATA_DIR / f"redcap_sample_cleanpos_{MIX_RATIO_NAME}.csv", index=False)
source_summary.to_csv(DATA_DIR / f"training_source_summary_cleanpos_{MIX_RATIO_NAME}.csv", index=False)

print("Saved:", train_pairs_path)
display(train_pairs.head(10))
print("STATUS: REDCAP CLEAN-POSITIVE TRAINING PAIRS BUILT")

In [ ]:
# =====================
# Cell 7 — Load init checkpoint as anti-drift reference and student initialization
# =====================

def get_first_module(st_model):
    return st_model._first_module()


def extract_state_dict(ckpt_path: Path) -> Dict[str, torch.Tensor]:
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    if isinstance(ckpt, dict):
        for k in ["model_state_dict", "state_dict", "model", "module"]:
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
    if isinstance(ckpt, dict):
        return ckpt
    raise ValueError(f"Unsupported checkpoint format: {type(ckpt)}")


def infer_lora_config_from_state(state_dict: Dict[str, torch.Tensor]):
    lora_a_keys = [k for k in state_dict if "lora_A" in k and hasattr(state_dict[k], "shape")]
    if not lora_a_keys:
        raise ValueError("No lora_A keys found in checkpoint. Is this a PEFT LoRA checkpoint?")
    sample = state_dict[lora_a_keys[0]]
    r = int(sample.shape[0])

    target_modules = []
    for tm in ["query", "value", "key", "dense"]:
        if any(f".{tm}.lora_A" in k or f".{tm}.lora_A." in k for k in lora_a_keys):
            target_modules.append(tm)
    if not target_modules:
        # Known Fan previous checkpoint had 96 LoRA keys, consistent with query+value on BGE-large.
        target_modules = ["query", "value"]

    print("Inferred LoRA r:", r)
    print("Inferred target_modules:", target_modules)
    return r, target_modules


def normalize_ckpt_key_candidates(k: str) -> List[str]:
    cands = [k]
    prefixes = [
        "encoder.", "model.", "module.", "0.", "0.auto_model.",
        "sentence_transformer.", "sentence_transformer.0.",
    ]
    for pref in prefixes:
        if k.startswith(pref):
            cands.append(k[len(pref):])

    replacements = [
        ("encoder.base_model.model.", "base_model.model."),
        ("model.base_model.model.", "base_model.model."),
        ("module.base_model.model.", "base_model.model."),
        ("auto_model.base_model.model.", "base_model.model."),
    ]
    for a, b in replacements:
        if k.startswith(a):
            cands.append(b + k[len(a):])

    # Also try adding base_model.model. if key begins with encoder.layer...
    if k.startswith("encoder.layer."):
        cands.append("base_model.model." + k)
    return list(dict.fromkeys(cands))


def load_fan_previous_sentence_transformer(base_model_name: str, ckpt_path: Path, device: torch.device, trainable: bool):
    state = extract_state_dict(ckpt_path)
    r, target_modules = infer_lora_config_from_state(state)

    st_model = SentenceTransformer(base_model_name, device=str(device))
    transformer = get_first_module(st_model)
    auto_model = transformer.auto_model

    lora_config = LoraConfig(
        r=r,
        lora_alpha=32 if r == 16 else max(2 * r, 16),
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="FEATURE_EXTRACTION",
    )
    peft_model = get_peft_model(auto_model, lora_config)

    model_sd = peft_model.state_dict()
    mapped = {}
    skipped_shape = []
    for k, v in state.items():
        if not hasattr(v, "shape"):
            continue
        for cand in normalize_ckpt_key_candidates(k):
            if cand in model_sd:
                if tuple(model_sd[cand].shape) == tuple(v.shape):
                    mapped[cand] = v
                else:
                    skipped_shape.append((k, cand, tuple(v.shape), tuple(model_sd[cand].shape)))
                break

    missing, unexpected = peft_model.load_state_dict(mapped, strict=False)
    transformer.auto_model = peft_model
    st_model.to(device)

    if not trainable:
        st_model.eval()
        for p in st_model.parameters():
            p.requires_grad = False
    else:
        st_model.train()
        # Freeze all non-LoRA params. PEFT usually does this already, but enforce it.
        for name, p in st_model.named_parameters():
            p.requires_grad = ("lora_" in name)

    lora_ckpt_keys = [k for k in state if "lora_" in k]
    matched_lora_keys = [k for k in mapped if "lora_" in k]
    print("Loaded checkpoint:", ckpt_path)
    print("Raw LoRA keys in checkpoint:", len(lora_ckpt_keys))
    print("Matched LoRA keys:", len(matched_lora_keys))
    print("Trainable params:", sum(p.numel() for p in st_model.parameters() if p.requires_grad))
    print("Total params:", sum(p.numel() for p in st_model.parameters()))

    if len(matched_lora_keys) == 0:
        raise RuntimeError("No LoRA keys matched. Stop: checkpoint was not loaded correctly.")
    return st_model

reference_model = load_fan_previous_sentence_transformer(
    BASE_MODEL_NAME,
    INIT_CHECKPOINT_PATH,
    device,
    trainable=False,
)

student_model = load_fan_previous_sentence_transformer(
    BASE_MODEL_NAME,
    INIT_CHECKPOINT_PATH,
    device,
    trainable=True,
)

print("STATUS: INIT REFERENCE AND STUDENT LOADED")

In [ ]:
# =====================
# Cell 8 — Embedding helper and init smoke test
# =====================

def encode_with_grad(st_model: SentenceTransformer, texts: List[str], device: torch.device):
    features = st_model.tokenize(list(texts))
    features = batch_to_device(features, device)
    out = st_model(features)
    emb = out["sentence_embedding"]
    emb = F.normalize(emb, p=2, dim=1)
    return emb

@torch.no_grad()
def encode_no_grad(st_model: SentenceTransformer, texts: List[str], device: torch.device):
    st_model.eval()
    emb = encode_with_grad(st_model, texts, device)
    return emb

sample_texts = train_pairs["anchor"].head(4).tolist()
with torch.no_grad():
    t_emb = encode_no_grad(reference_model, sample_texts, device)
    s_emb = encode_no_grad(student_model, sample_texts, device)
    cos = (t_emb * s_emb).sum(dim=1).detach().cpu().numpy()

print("reference/student initial cosine on sample anchors:", cos)
print("mean:", cos.mean())
assert cos.mean() > 0.999, "Student should start identical/near-identical to reference. Check checkpoint loading."
print("STATUS: REFERENCE/STUDENT SMOKE TEST PASSED")

In [ ]:
# Cell 9 — Dataset and dataloader
class PairTextDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.anchor = df["anchor"].astype(str).tolist()
        self.positive = df["positive"].astype(str).tolist()
        self.source_type = df["source_type"].astype(str).tolist() if "source_type" in df.columns else ["unknown"] * len(df)
        self.is_targeted_option_row = df["is_targeted_option_row"].fillna(False).astype(bool).tolist() if "is_targeted_option_row" in df.columns else [False] * len(df)

    def __len__(self):
        return len(self.anchor)

    def __getitem__(self, idx):
        return {
            "anchor": self.anchor[idx],
            "positive": self.positive[idx],
            "source_type": self.source_type[idx],
            "is_targeted_option_row": self.is_targeted_option_row[idx],
        }


def collate_pairs(batch):
    return {
        "anchor": [x["anchor"] for x in batch],
        "positive": [x["positive"] for x in batch],
        "source_type": [x["source_type"] for x in batch],
        "is_targeted_option_row": [x["is_targeted_option_row"] for x in batch],
    }

train_dataset = PairTextDataset(train_pairs)
train_loader = DataLoader(
    train_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_pairs,
    drop_last=True,
)

steps_per_epoch = len(train_loader)
max_train_steps = max(1, math.ceil(steps_per_epoch * MAX_EPOCH_FRACTION))
print("train examples:", len(train_dataset))
print("batch size:", PER_DEVICE_BATCH_SIZE)
print("steps_per_epoch:", steps_per_epoch)
print("max_train_steps:", max_train_steps)
print("targeted option examples:", int(pd.Series(train_dataset.is_targeted_option_row).sum()))


In [ ]:
# Cell 10 — Train with contrastive loss + anti-drift
# Loss:
#   contrastive_loss = MNRL-style in-batch cross entropy
#   anti_drift_loss = MSE(student embeddings, init checkpoint embeddings)

optimizer = torch.optim.AdamW(
    [p for p in student_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

student_model.train()
reference_model.eval()

train_log = []
step = 0
optimizer.zero_grad(set_to_none=True)

pbar = tqdm(train_loader, total=max_train_steps, desc="REDCap 9to1 clean-positive train")
for batch_idx, batch in enumerate(pbar):
    if step >= max_train_steps:
        break

    anchors = batch["anchor"]
    positives = batch["positive"]

    s_anchor = encode_with_grad(student_model, anchors, device)
    s_pos = encode_with_grad(student_model, positives, device)

    logits = torch.matmul(s_anchor, s_pos.T) / TEMPERATURE
    labels = torch.arange(logits.size(0), device=device)
    loss_a2p = F.cross_entropy(logits, labels)
    loss_p2a = F.cross_entropy(logits.T, labels)
    contrastive_loss = 0.5 * (loss_a2p + loss_p2a)

    anti_drift_loss = torch.tensor(0.0, device=device)
    if ANTI_DRIFT_DISTILL_WEIGHT > 0:
        with torch.no_grad():
            t_anchor = encode_no_grad(reference_model, anchors, device)
            t_pos = encode_no_grad(reference_model, positives, device)
        anti_drift_loss = 0.5 * (F.mse_loss(s_anchor, t_anchor) + F.mse_loss(s_pos, t_pos))

    loss = contrastive_loss + ANTI_DRIFT_DISTILL_WEIGHT * anti_drift_loss
    loss = loss / GRAD_ACCUM_STEPS
    loss.backward()

    if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_([p for p in student_model.parameters() if p.requires_grad], MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    step += 1

    log_row = {
        "step": step,
        "loss": float(loss.detach().cpu().item() * GRAD_ACCUM_STEPS),
        "contrastive_loss": float(contrastive_loss.detach().cpu().item()),
        "anti_drift_loss": float(anti_drift_loss.detach().cpu().item()),
        "lr": LEARNING_RATE,
        "batch_clean_n": int(sum(1 for x in batch["source_type"] if x == "clean")),
        "batch_redcap_n": int(sum(1 for x in batch["source_type"] if "redcap" in x)),
        "batch_targeted_option_n": int(sum(1 for x in batch.get("is_targeted_option_row", []) if bool(x))),
    }
    train_log.append(log_row)
    pbar.set_postfix({
        "loss": f"{log_row['loss']:.4f}",
        "ctr": f"{log_row['contrastive_loss']:.4f}",
        "drift": f"{log_row['anti_drift_loss']:.6f}",
    })

train_log_df = pd.DataFrame(train_log)
train_log_path = OUT_DIR / "train_log.csv"
train_log_df.to_csv(train_log_path, index=False)
print("Saved train log:", train_log_path)
display(train_log_df.tail())
print("STATUS: REDCAP 9TO1 CLEAN-POSITIVE TRAINING DONE")

In [ ]:
# =====================
# Cell 11 — Post-training drift check vs init checkpoint reference
# =====================
# We want drift to be small. If cosine is too low, LR/epoch/REDCap ratio is too aggressive.

student_model.eval()
reference_model.eval()

sample_n = min(512, len(train_pairs))
sample_df = train_pairs.sample(n=sample_n, random_state=SEED)
texts = sample_df["anchor"].tolist() + sample_df["positive"].tolist()

cos_vals = []
for i in tqdm(range(0, len(texts), PER_DEVICE_BATCH_SIZE), desc="drift check"):
    chunk = texts[i:i+PER_DEVICE_BATCH_SIZE]
    with torch.no_grad():
        t = encode_no_grad(reference_model, chunk, device)
        s = encode_no_grad(student_model, chunk, device)
        cos_vals.extend((t * s).sum(dim=1).detach().cpu().numpy().tolist())

cos_vals = np.array(cos_vals)
drift_summary = pd.DataFrame([{
    "n_texts": len(cos_vals),
    "reference_student_cos_mean": float(cos_vals.mean()),
    "reference_student_cos_median": float(np.median(cos_vals)),
    "reference_student_cos_p05": float(np.percentile(cos_vals, 5)),
    "reference_student_cos_min": float(cos_vals.min()),
    "mix_ratio": MIX_RATIO_NAME,
    "learning_rate": LEARNING_RATE,
    "anti_drift_distill_weight": ANTI_DRIFT_DISTILL_WEIGHT,
    "max_epoch_fraction": MAX_EPOCH_FRACTION,
}])

drift_path = OUT_DIR / "reference_student_drift_summary.csv"
drift_summary.to_csv(drift_path, index=False)
print("Saved:", drift_path)
display(drift_summary)

if drift_summary["reference_student_cos_mean"].iloc[0] < 0.995:
    print("WARNING: Mean cosine drift is relatively large. Consider lowering LR, reducing epoch fraction, or increasing ANTI_DRIFT_DISTILL_WEIGHT.")
else:
    print("Drift looks conservative.")

In [ ]:
# Cell 12 — Save REDCap 9:1 checkpoint
FINAL_CKPT_PATH = CKPT_DIR / "biencoder_lora_final.pt"
FINAL_ST_PATH = CKPT_DIR / "sentence_transformer_final"

student_model.eval()
auto_model = student_model._first_module().auto_model
state_dict = {k: v.detach().cpu() for k, v in auto_model.state_dict().items()}

save_obj = {
    "model_state_dict": state_dict,
    "base_model_name": BASE_MODEL_NAME,
    "init_checkpoint_path": str(INIT_CHECKPOINT_PATH),
    "training_strategy": "init_checkpoint_clean_nci_plus_filtered_redcap_clean_positive_9to1",
    "mix_ratio_name": MIX_RATIO_NAME,
    "clean_to_redcap_ratio": CLEAN_TO_REDCAP_RATIO,
    "targeted_redcap_min_fraction": TARGETED_REDCAP_MIN_FRACTION,
    "learning_rate": LEARNING_RATE,
    "anti_drift_distill_weight": ANTI_DRIFT_DISTILL_WEIGHT,
    "temperature": TEMPERATURE,
    "max_epoch_fraction": MAX_EPOCH_FRACTION,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "encode_batch_size": ENCODE_BATCH_SIZE,
    "seed": SEED,
    "train_pairs_path": str(train_pairs_path),
    "redcap_filter_audit_path": str(DATA_DIR / "redcap_synthetic_filter_audit.csv"),
}

torch.save(save_obj, FINAL_CKPT_PATH)
student_model.save(str(FINAL_ST_PATH))

print("Saved LoRA checkpoint:", FINAL_CKPT_PATH)
print("Saved SentenceTransformer folder:", FINAL_ST_PATH)
print("checkpoint exists:", FINAL_CKPT_PATH.exists())
print("STATUS: REDCAP 9TO1 CHECKPOINT SAVED")

In [ ]:
# Cell 13 — Load previous fixed-benchmark rerun outputs for historical comparison
from pathlib import Path

def discover_previous_rerun_dir() -> Path:
    if PREVIOUS_RERUN_DIR is not None:
        p = Path(PREVIOUS_RERUN_DIR)
        if not p.exists():
            raise FileNotFoundError(f"PREVIOUS_RERUN_DIR does not exist: {p}")
        return p
    candidates = []
    for p in sorted(Path("/").glob(PREVIOUS_RERUN_GLOB.lstrip("/"))):
        if (p / "benchmark_query_texts_embedded.csv").exists() and (p / "target_texts_embedded.csv").exists():
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(
            "Could not auto-discover previous rerun dir. Set PREVIOUS_RERUN_DIR in Cell 1. "
            "Need files: benchmark_query_texts_embedded.csv and target_texts_embedded.csv."
        )
    # newest modified directory wins
    candidates.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]

PREV_DIR = discover_previous_rerun_dir()
print("Using previous rerun dir:", PREV_DIR)

QUERY_TEXTS_PATH = PREV_DIR / "benchmark_query_texts_embedded.csv"
TARGET_TEXTS_PATH = PREV_DIR / "target_texts_embedded.csv"
OLD_OVERALL_PATH = PREV_DIR / "final_overall_model_comparison.csv"
OLD_METRICS_PATH = PREV_DIR / "full_rerun_metrics_per_row.csv"

assert QUERY_TEXTS_PATH.exists(), QUERY_TEXTS_PATH
assert TARGET_TEXTS_PATH.exists(), TARGET_TEXTS_PATH
assert OLD_OVERALL_PATH.exists() or OLD_METRICS_PATH.exists(), "Need old overall or old metrics CSV from previous rerun."

query_df = pd.read_csv(QUERY_TEXTS_PATH, low_memory=False)
target_df = pd.read_csv(TARGET_TEXTS_PATH, low_memory=False)
print("query_df:", query_df.shape)
print("target_df:", target_df.shape)
print("query columns:", query_df.columns.tolist())
print("target columns:", target_df.columns.tolist())

def choose_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}. Existing columns: {list(df.columns)[:50]}")

QUERY_TEXT_COL = choose_col(query_df, ["query_text_field_code_first_typed", "query_text"])
TARGET_TEXT_COL = choose_col(target_df, ["target_text_sde_code_name_desc_val", "target_text"])
print("QUERY_TEXT_COL:", QUERY_TEXT_COL)
print("TARGET_TEXT_COL:", TARGET_TEXT_COL)

# Rename query_text for downstream consistency.
eval_df = query_df.copy()
eval_df["query_text"] = eval_df[QUERY_TEXT_COL].astype(str)
target_work = target_df.copy()
target_work["target_text"] = target_work[TARGET_TEXT_COL].astype(str)

print("eval rows:", len(eval_df))
print("target rows:", len(target_work))

In [ ]:
# Cell 14 — Evaluation utilities for the new REDCap 9:1 model
TOP_K_VALUES = [1, 5, 10]
INITIAL_RETRIEVAL_K = 100
BM25_RETRIEVAL_K = 100
FINAL_RERANK_K = 50
EMBEDDING_WEIGHT = 0.70
BM25_WEIGHT = 0.30
BM25_K1 = 1.2
BM25_B = 0.75
NEW_MODEL_KEY = "redcap_cleanpos_9to1_antidrift"
NEW_MODEL_NAME = "REDCap clean-positive 9to1 anti-drift"


def stringify(x) -> str:
    if pd.isna(x):
        return ""
    return str(x)


def normalize_id_for_eval(x) -> str:
    return re.sub(r"\s+", "", stringify(x)).strip().lower()


def normalize_name_for_eval(x) -> str:
    s = stringify(x).strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def tokenize(s: str) -> list[str]:
    s = stringify(s).lower()
    return re.findall(r"[a-z0-9]+", s)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    denom = np.linalg.norm(x, axis=1, keepdims=True)
    denom = np.maximum(denom, 1e-12)
    return x / denom


class SimpleBM25:
    def __init__(self, docs: list[str], k1: float = BM25_K1, b: float = BM25_B):
        self.docs = docs
        self.doc_tokens = [tokenize(d) for d in docs]
        self.n_docs = len(self.doc_tokens)
        self.k1 = k1
        self.b = b
        self.doc_lens = np.array([len(toks) for toks in self.doc_tokens], dtype=np.float32)
        self.avgdl = float(np.mean(self.doc_lens)) if len(self.doc_lens) else 0.0
        df: Counter[str] = Counter()
        postings: dict[str, list[tuple[int, int]]] = {}
        for i, toks in enumerate(self.doc_tokens):
            counts = Counter(toks)
            for term, count in counts.items():
                df[term] += 1
                postings.setdefault(term, []).append((i, count))
        self.idf = {term: math.log((self.n_docs - freq + 0.5) / (freq + 0.5) + 1.0) for term, freq in df.items()}
        self.postings = postings

    def topk(self, query: str, k: int) -> tuple[np.ndarray, np.ndarray]:
        scores: dict[int, float] = {}
        for term in tokenize(query):
            for doc_idx, freq in self.postings.get(term, []):
                dl = self.doc_lens[doc_idx]
                denom = freq + self.k1 * (1.0 - self.b + self.b * dl / max(self.avgdl, 1e-9))
                score = self.idf.get(term, 0.0) * (freq * (self.k1 + 1.0)) / max(denom, 1e-9)
                scores[doc_idx] = scores.get(doc_idx, 0.0) + score
        if not scores:
            return np.array([], dtype=int), np.array([], dtype=np.float32)
        items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
        return np.array([i for i, _ in items], dtype=int), np.array([s for _, s in items], dtype=np.float32)


def minmax_dict(score_dict: dict[int, float]) -> dict[int, float]:
    if not score_dict:
        return {}
    vals = np.array(list(score_dict.values()), dtype=np.float32)
    lo = float(np.min(vals)); hi = float(np.max(vals))
    if hi - lo < 1e-12:
        return {k: 1.0 for k in score_dict}
    return {k: float((v - lo) / (hi - lo)) for k, v in score_dict.items()}


def topk_search(query_emb: np.ndarray, target_emb: np.ndarray, k: int, batch_size: int = 128) -> tuple[np.ndarray, np.ndarray]:
    query_emb = l2_normalize(query_emb)
    target_emb = l2_normalize(target_emb)
    k = min(k, target_emb.shape[0])
    all_idx = []
    all_scores = []
    for start in range(0, query_emb.shape[0], batch_size):
        scores = query_emb[start:start + batch_size] @ target_emb.T
        idx = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
        rows = np.arange(scores.shape[0])[:, None]
        order = np.argsort(-scores[rows, idx], axis=1)
        idx = idx[rows, order]
        all_idx.append(idx.astype(int))
        all_scores.append(scores[rows, idx].astype(np.float32))
    return np.vstack(all_idx), np.vstack(all_scores)


def hybrid_union_rerank_details(query_texts: list[str], query_emb: np.ndarray, target_emb: np.ndarray, initial_idx: np.ndarray, bm25: SimpleBM25):
    query_emb = l2_normalize(query_emb)
    target_emb = l2_normalize(target_emb)
    all_ranked = []
    for i, q in enumerate(tqdm(query_texts, desc="hybrid_union_rerank_top50")):
        embed_candidates = [int(x) for x in initial_idx[i][:INITIAL_RETRIEVAL_K]]
        embed_raw = (target_emb[np.array(embed_candidates, dtype=int)] @ query_emb[i]).astype(np.float32) if embed_candidates else np.array([], dtype=np.float32)
        emb_dict = {int(ci): float(s) for ci, s in zip(embed_candidates, embed_raw)}
        dense_rank = {int(ci): r + 1 for r, ci in enumerate(embed_candidates)}

        bm25_idx, bm25_scores = bm25.topk(q, BM25_RETRIEVAL_K)
        bm25_dict = {int(ci): float(s) for ci, s in zip(bm25_idx, bm25_scores)}
        bm25_rank = {int(ci): r + 1 for r, ci in enumerate(bm25_idx.tolist())}

        emb_norm = minmax_dict(emb_dict)
        bm25_norm = minmax_dict(bm25_dict)
        union = sorted(set(emb_dict) | set(bm25_dict))
        scored = []
        for ci in union:
            e = emb_norm.get(ci, 0.0)
            b = bm25_norm.get(ci, 0.0)
            hybrid = EMBEDDING_WEIGHT * e + BM25_WEIGHT * b
            scored.append({
                "target_index": int(ci),
                "dense_score_raw": emb_dict.get(ci, np.nan),
                "dense_score_norm": e,
                "dense_rank": dense_rank.get(ci, np.nan),
                "bm25_score_raw": bm25_dict.get(ci, np.nan),
                "bm25_score_norm": b,
                "bm25_rank": bm25_rank.get(ci, np.nan),
                "hybrid_score": float(hybrid),
            })
        scored.sort(key=lambda x: x["hybrid_score"], reverse=True)
        for rank, item in enumerate(scored, start=1):
            item["candidate_rank"] = rank
        all_ranked.append(scored)
    return all_ranked


def candidate_is_gold(row: pd.Series, target_row: pd.Series) -> bool:
    gold_ids = {normalize_id_for_eval(row.get("element_name"))} - {""}
    gold_names = {normalize_name_for_eval(row.get("element_title"))} - {""}
    target_ids = {normalize_id_for_eval(target_row.get("sde_id"))} - {""}
    target_names = {
        normalize_name_for_eval(target_row.get("sde_name")),
        normalize_name_for_eval(target_row.get("sde_title")),
    } - {""}
    return bool((gold_ids & target_ids) or (gold_names & target_names))


def summarize_metrics(metrics: pd.DataFrame, model_key: str, model_name: str) -> pd.DataFrame:
    rows = []
    for bench, g in list(metrics.groupby("benchmark", dropna=False)) + [("ALL_COMBINED", metrics)]:
        out = {
            "model_key": model_key,
            "model": model_name,
            "benchmark": bench,
            "n_eval_rows": len(g),
        }
        for k in TOP_K_VALUES:
            hits = int(g[f"match_at_{k}"].sum())
            out[f"top{k}_hits"] = hits
            out[f"top{k}_accuracy"] = hits / max(len(g), 1)
        rows.append(out)
    return pd.DataFrame(rows)


def build_candidate_audit_and_metrics(run_df: pd.DataFrame, target_df_work: pd.DataFrame, ranked_details: list[list[dict[str, Any]]]):
    metric_rows = []
    audit_rows = []
    for i, row in run_df.reset_index(drop=True).iterrows():
        ranked = ranked_details[i]
        out = row.to_dict()
        out["experiment_name"] = "redcap_cleanpos_9to1_field_code_first_typed_same_eval_pipeline"
        out["experiment_group"] = "redcap_cleanpos_9to1_new_benchmark"
        out["model_key"] = NEW_MODEL_KEY
        out["model"] = NEW_MODEL_NAME
        out["backend"] = "sentence_transformers_checkpoint"
        out["query_representation"] = "field_code_first_typed"
        out["target_representation"] = "sde_code_name_desc_val"
        out["retrieval_stage"] = "hybrid_union_rerank"

        gold_rank = np.nan
        for item in ranked:
            target_row = target_df_work.iloc[int(item["target_index"])]
            if candidate_is_gold(row, target_row):
                gold_rank = int(item["candidate_rank"])
                break
        out["gold_rank_in_full_union"] = gold_rank
        for k in TOP_K_VALUES:
            topk_items = ranked[:k]
            matches, ids, names, scores = [], [], [], []
            for item in topk_items:
                target_row = target_df_work.iloc[int(item["target_index"])]
                ids.append(stringify(target_row.get("sde_id")))
                names.append(stringify(target_row.get("sde_name")))
                scores.append(float(item["hybrid_score"]))
                matches.append(candidate_is_gold(row, target_row))
            out[f"top_{k}_sde_ids"] = " || ".join(ids)
            out[f"top_{k}_sde_names"] = " || ".join(names)
            out[f"top_{k}_scores"] = " || ".join(f"{s:.6f}" for s in scores)
            out[f"match_at_{k}"] = bool(any(matches))
        out["hit_top50"] = bool(pd.notna(gold_rank) and float(gold_rank) <= FINAL_RERANK_K)
        metric_rows.append(out)

        for item in ranked[:FINAL_RERANK_K]:
            target_row = target_df_work.iloc[int(item["target_index"])]
            audit_rows.append({
                "model_key": NEW_MODEL_KEY,
                "benchmark": row.get("benchmark"),
                "row_uid": row.get("row_uid"),
                "source_row_number": row.get("source_row_number"),
                "field_name": row.get("field_name"),
                "field_type": row.get("field_type"),
                "field_title": row.get("field_title"),
                "field_enumLabels": row.get("field_enumLabels"),
                "field_constraints": row.get("field_constraints"),
                "field_description": row.get("field_description"),
                "gold_element_name": row.get("element_name"),
                "gold_element_title": row.get("element_title"),
                "query_text": row.get("query_text", ""),
                "candidate_rank": item["candidate_rank"],
                "candidate_sde_id": target_row.get("sde_id"),
                "candidate_sde_name": target_row.get("sde_name"),
                "candidate_sde_title": target_row.get("sde_title"),
                "candidate_text": target_row.get("target_text"),
                "is_gold": candidate_is_gold(row, target_row),
                "hybrid_score": item["hybrid_score"],
                "dense_score_raw": item["dense_score_raw"],
                "dense_rank": item["dense_rank"],
                "bm25_score_raw": item["bm25_score_raw"],
                "bm25_rank": item["bm25_rank"],
            })
    metrics = pd.DataFrame(metric_rows)
    audit = pd.DataFrame(audit_rows)
    summary = summarize_metrics(metrics, NEW_MODEL_KEY, NEW_MODEL_NAME)
    return summary, metrics, audit


@torch.no_grad()
def encode_sentence_transformer_numpy(st_model: SentenceTransformer, texts: list[str], batch_size: int = ENCODE_BATCH_SIZE) -> np.ndarray:
    st_model.eval()
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="encode_new_model"):
        batch = texts[i:i+batch_size]
        emb = encode_no_grad(st_model, batch, device)
        out.append(emb.detach().cpu().numpy().astype(np.float32))
    return l2_normalize(np.vstack(out))

In [ ]:
# Cell 15 — Evaluate the new REDCap 9:1 model on the same fixed benchmark rows/targets
query_texts = eval_df["query_text"].fillna("").astype(str).tolist()
target_texts = target_work["target_text"].fillna("").astype(str).tolist()

print("Encoding target texts with new REDCap 9:1 model...")
new_target_emb = encode_sentence_transformer_numpy(student_model, target_texts, batch_size=ENCODE_BATCH_SIZE)
print("Encoding query texts with new REDCap 9:1 model...")
new_query_emb = encode_sentence_transformer_numpy(student_model, query_texts, batch_size=ENCODE_BATCH_SIZE)

np.save(EVAL_DIR / "new_model_target_embeddings.npy", new_target_emb)
np.save(EVAL_DIR / "new_model_query_embeddings.npy", new_query_emb)
print("Saved new model embeddings under:", EVAL_DIR)

print("Dense retrieve...")
initial_idx, initial_scores = topk_search(new_query_emb, new_target_emb, k=min(INITIAL_RETRIEVAL_K, len(target_work)))

print("BM25 index...")
bm25 = SimpleBM25(target_texts)

print("Hybrid union rerank...")
ranked_details = hybrid_union_rerank_details(query_texts, new_query_emb, new_target_emb, initial_idx, bm25)

new_summary, new_metrics, new_audit = build_candidate_audit_and_metrics(eval_df, target_work, ranked_details)
new_summary_path = EVAL_DIR / "new_model_summary_by_benchmark.csv"
new_metrics_path = EVAL_DIR / "new_model_metrics_per_row.csv"
new_audit_path = EVAL_DIR / "new_model_all_rows_top50_candidates.csv"
new_top10_miss_path = EVAL_DIR / "new_model_top10_miss_top50_candidates.csv"

new_summary.to_csv(new_summary_path, index=False)
new_metrics.to_csv(new_metrics_path, index=False)
new_audit.to_csv(new_audit_path, index=False)
miss_uids = set(new_metrics.loc[~new_metrics["match_at_10"], "row_uid"].astype(str))
new_audit[new_audit["row_uid"].astype(str).isin(miss_uids)].to_csv(new_top10_miss_path, index=False)

print("Saved:", new_summary_path)
print("Saved:", new_metrics_path)
print("Saved:", new_audit_path)
print("Saved:", new_top10_miss_path)
display(new_summary[new_summary["benchmark"].eq("ALL_COMBINED")])

In [ ]:
# Cell 16 — Compare new model with historical REDCap v3, BGE, OpenAI, and CTDS base results
# Old models are read from previous CSV outputs; this cell does not re-embed them.

def normalize_old_summary_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Accept either top1_accuracy or accuracy_at_1 style.
    rename = {}
    for k in TOP_K_VALUES:
        if f"accuracy_at_{k}" in out.columns and f"top{k}_accuracy" not in out.columns:
            rename[f"accuracy_at_{k}"] = f"top{k}_accuracy"
        if f"hits_at_{k}" in out.columns and f"top{k}_hits" not in out.columns:
            rename[f"hits_at_{k}"] = f"top{k}_hits"
    out = out.rename(columns=rename)
    return out


def compute_old_summary_from_metrics(path: Path) -> pd.DataFrame:
    m = pd.read_csv(path, low_memory=False)
    rows = []
    for (model_key, model, bench), g in m.groupby(["model_key", "model", "benchmark"], dropna=False):
        row = {"model_key": model_key, "model": model, "benchmark": bench, "n_eval_rows": len(g)}
        for k in TOP_K_VALUES:
            row[f"top{k}_hits"] = int(g[f"match_at_{k}"].sum())
            row[f"top{k}_accuracy"] = row[f"top{k}_hits"] / max(len(g), 1)
        rows.append(row)
    for (model_key, model), g in m.groupby(["model_key", "model"], dropna=False):
        row = {"model_key": model_key, "model": model, "benchmark": "ALL_COMBINED", "n_eval_rows": len(g)}
        for k in TOP_K_VALUES:
            row[f"top{k}_hits"] = int(g[f"match_at_{k}"].sum())
            row[f"top{k}_accuracy"] = row[f"top{k}_hits"] / max(len(g), 1)
        rows.append(row)
    return pd.DataFrame(rows)

if OLD_OVERALL_PATH.exists():
    old_summary_raw = pd.read_csv(OLD_OVERALL_PATH, low_memory=False)
    old_summary = normalize_old_summary_columns(old_summary_raw)
else:
    old_summary = compute_old_summary_from_metrics(OLD_METRICS_PATH)

old_summary["model_key"] = old_summary["model_key"].astype(str)
old_summary["model"] = old_summary["model"].astype(str)
old_summary["benchmark"] = old_summary["benchmark"].astype(str)

# Keep only requested old models.
patterns = {
    "redcap_v3_old": r"fan_redcap_v3|redcap_v3|targeted_option_distill",
    "bge_raw_old": r"bge_raw|BGE raw|BAAI/bge",
    "openai_raw_old": r"openai_raw|OpenAI raw|text-embedding",
    "ctds_base_raw_old": r"ctds_base|CTDS base|uc-ctds/bge-large-en-v1.5-bio-mapping",
}
keep_mask = pd.Series(False, index=old_summary.index)
old_summary["comparison_bucket"] = ""
for bucket, pat in patterns.items():
    m = old_summary["model_key"].str.contains(pat, case=False, regex=True, na=False) | old_summary["model"].str.contains(pat, case=False, regex=True, na=False)
    old_summary.loc[m, "comparison_bucket"] = bucket
    keep_mask = keep_mask | m
old_keep = old_summary[keep_mask].copy()

# If multiple rows match same bucket/benchmark, keep the best known row by top10 then top5 then top1.
old_keep = old_keep.sort_values(["benchmark", "comparison_bucket", "top10_accuracy", "top5_accuracy", "top1_accuracy"], ascending=[True, True, False, False, False])
old_keep = old_keep.drop_duplicates(["benchmark", "comparison_bucket"], keep="first")

new_keep = new_summary.copy()
new_keep["comparison_bucket"] = "new_redcap_cleanpos_9to1"

comparison = pd.concat([old_keep, new_keep], ignore_index=True, sort=False)
comparison_cols = [
    "comparison_bucket", "model_key", "model", "benchmark", "n_eval_rows",
    "top1_hits", "top1_accuracy", "top5_hits", "top5_accuracy", "top10_hits", "top10_accuracy"
]
comparison = comparison[[c for c in comparison_cols if c in comparison.columns]].copy()
comparison = comparison.sort_values(["benchmark", "top10_accuracy", "top5_accuracy", "top1_accuracy"], ascending=[True, False, False, False])

comparison_path = EVAL_DIR / "comparison_new_vs_old_requested_models_by_benchmark.csv"
comparison.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)

print("ALL_COMBINED comparison:")
display(comparison[comparison["benchmark"].eq("ALL_COMBINED")].sort_values(["top10_accuracy", "top5_accuracy", "top1_accuracy"], ascending=False))

print("By benchmark comparison:")
display(comparison[~comparison["benchmark"].eq("ALL_COMBINED")].head(50))

# Also write a compact delta table against old REDCap v3 if available.
all_combined = comparison[comparison["benchmark"].eq("ALL_COMBINED")].copy()
base = all_combined[all_combined["comparison_bucket"].eq("redcap_v3_old")]
new = all_combined[all_combined["comparison_bucket"].eq("new_redcap_cleanpos_9to1")]
if len(base) and len(new):
    delta = pd.DataFrame([{
        "metric": f"top{k}",
        "old_redcap_v3_hits": int(base.iloc[0].get(f"top{k}_hits", np.nan)),
        "new_hits": int(new.iloc[0].get(f"top{k}_hits", np.nan)),
        "delta_hits": int(new.iloc[0].get(f"top{k}_hits", 0) - base.iloc[0].get(f"top{k}_hits", 0)),
        "old_redcap_v3_accuracy": float(base.iloc[0].get(f"top{k}_accuracy", np.nan)),
        "new_accuracy": float(new.iloc[0].get(f"top{k}_accuracy", np.nan)),
        "delta_accuracy": float(new.iloc[0].get(f"top{k}_accuracy", 0) - base.iloc[0].get(f"top{k}_accuracy", 0)),
    } for k in TOP_K_VALUES])
    delta_path = EVAL_DIR / "delta_new_vs_old_redcap_v3_all_combined.csv"
    delta.to_csv(delta_path, index=False)
    print("Saved:", delta_path)
    display(delta)
else:
    print("Could not find old REDCap v3 row for delta table. Check comparison buckets.")

In [ ]:
# Cell 17 — Save run manifest
manifest = {
    "out_dir": str(OUT_DIR),
    "data_dir": str(DATA_DIR),
    "ckpt_dir": str(CKPT_DIR),
    "eval_dir": str(EVAL_DIR),
    "init_checkpoint_path": str(INIT_CHECKPOINT_PATH),
    "base_model_name": BASE_MODEL_NAME,
    "clean_full_split_path": str(CLEAN_FULL_SPLIT_PATH),
    "augmented_train_plus_redcap_path": str(AUGMENTED_TRAIN_PLUS_REDCAP_PATH),
    "previous_rerun_dir": str(PREV_DIR),
    "query_texts_path": str(QUERY_TEXTS_PATH),
    "target_texts_path": str(TARGET_TEXTS_PATH),
    "final_checkpoint_path": str(FINAL_CKPT_PATH),
    "final_sentence_transformer_path": str(FINAL_ST_PATH),
    "train_pairs_path": str(train_pairs_path),
    "train_log_path": str(train_log_path),
    "new_summary_path": str(new_summary_path),
    "new_metrics_path": str(new_metrics_path),
    "new_audit_path": str(new_audit_path),
    "comparison_path": str(comparison_path),
    "clean_pairs_n": int(len(clean_pairs)),
    "redcap_raw_n": int(len(redcap_raw)),
    "redcap_keep_n": int(len(redcap_keep)),
    "redcap_sample_n": int(len(redcap_sample)),
    "train_pairs_n": int(len(train_pairs)),
    "clean_to_redcap_ratio": CLEAN_TO_REDCAP_RATIO,
    "learning_rate": LEARNING_RATE,
    "anti_drift_distill_weight": ANTI_DRIFT_DISTILL_WEIGHT,
    "max_epoch_fraction": MAX_EPOCH_FRACTION,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "encode_batch_size": ENCODE_BATCH_SIZE,
}
manifest_path = OUT_DIR / "run_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))
print("Saved:", manifest_path)
print("STATUS: NOTEBOOK DONE")